# Tutorial 7: GraphRAG, Retrieval over a Knowledge Graph

Tutorial 6 ended with a planner agent answering "where can I find the person who gave the KV cache lecture?" by
searching twice: once to learn the name, once to find the room. Every time someone asks a question like that, the
system rediscovers the same connection from raw text.

**GraphRAG** does that work once, at indexing time. An LLM reads the documents and extracts the entities in them
(people, courses, rooms) and the relations between them, and stores the result as a **knowledge graph**. At question
time you retrieve *facts and connections* from the graph instead of, or as well as, chunks of text.

By the end you will be able to:

- explain what a **knowledge graph** stores, and which questions it answers that vector search cannot;
- build and query a graph with **NetworkX**: triples, neighbours, paths;
- **extract** a graph from text with an LLM and a typed schema, and say where extraction goes wrong;
- do basic **entity resolution**, and explain why it decides the quality of the graph;
- answer questions with **local search** (entity linking and a k-hop subgraph);
- give an agent **graph tools** and let it explore;
- answer whole-corpus questions with **global search** (communities, reports, map-reduce);
- build a **hybrid** agent that chooses between text and graph retrieval, and decide when a graph is worth its cost.

## 0. Mental model: chunks versus triples

Vector RAG stores text and finds it by similarity. A knowledge graph stores **triples**, each one a single fact:

```text
(subject) ──[RELATION]──→ (object)

(Sara Karimi) ──[LEADS_SESSION_ON]──→ (KV cache)
(Sara Karimi) ──[HOLDS_OFFICE_HOURS_IN]──→ (room 305)
```

Facts that share an entity join up, so a question that needed two searches in tutorial 6 becomes a short walk:

```text
(KV cache) ←─LEADS_SESSION_ON── (Sara Karimi) ──HOLDS_OFFICE_HOURS_IN──→ (room 305)
```

| | Vector RAG | GraphRAG |
|---|---|---|
| Stores | Chunks of text and their embeddings | Entities and typed relations, with a link back to the source text |
| Finds | Text that *sounds like* the question | Facts *connected to* the entities in the question |
| Good at | Lookups, wording, nuance, anything written in prose | Multi-hop links, "who/which ... the most", whole-corpus overviews |
| Bad at | Connections spread over documents, counting, "summarise everything" | Facts that are not about entities (rules, explanations), anything extraction missed |
| Indexing cost | One embedding per chunk, cheap | One or more LLM calls per chunk, expensive |

The last row is the real trade. A graph moves work from question time to indexing time, and the work is done by an
LLM, so it can be wrong in ways that are invisible until a question fails. Most of this notebook is about doing that
step carefully.

## 1. Setup

Same environment as tutorial 6, plus NetworkX and Matplotlib for the graph:

```bash
pip install "langchain>=1.0" langchain-core langchain-groq langgraph python-dotenv
pip install langchain-huggingface sentence-transformers networkx matplotlib
```

and the same `.env` beside this notebook:

```text
GROQ_API_KEY=your-key-here
```

**NetworkX** keeps the whole graph in memory as Python objects. That is ideal for learning and fine for a few thousand
nodes. Section 11 shows what changes with a graph database such as Neo4j.

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()
assert os.getenv("GROQ_API_KEY"), "GROQ_API_KEY not found, see tutorial 3, section 1."

from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage
from langchain_core.documents import Document
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_huggingface import HuggingFaceEmbeddings

# Extraction quality matters a lot in this notebook. If results look poor, or you hit rate limits,
# try "openai/gpt-oss-120b" here.
model = ChatGroq(model="openai/gpt-oss-20b", temperature=0)
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

LIMITS = {"recursion_limit": 30}


def show_trace(messages, width=160):
    """Print one line per message: who spoke, and either the tool calls or the text."""
    for m in messages:
        kind = getattr(m, "name", None) or type(m).__name__
        calls = getattr(m, "tool_calls", None)
        if calls:
            body = "→ " + ", ".join(f"{c['name']}({c['args']})" for c in calls)
        else:
            body = str(m.content).replace("\n", " ")
        print(f"{kind:>14}: {body[:width]}")


def ask(agent, prompt):
    return agent.invoke({"messages": [HumanMessage(prompt)]}, config=LIMITS)


print("model and embeddings ready")

## 2. The corpus: the handbook, with a third course

The same made-up handbook as tutorial 6, plus four documents about a computer vision course. The new documents add
connections that cross courses: the same person manages the cluster both courses use, and two research groups link
the teaching assistants to the instructors.

The cell also builds tutorial 6's vector retriever over the same documents, so every GraphRAG answer below can be
compared with plain vector RAG on the same question.

In [ ]:
HANDBOOK = {
    "llm-syllabus": "LLM-501 (Large Language Models) is taught by Dr. Navid Rahimi. Grading: final project 40%, "
                    "final exam 35%, midterm 25%.",
    "llm-schedule": "In LLM-501, the week 9 session on the KV cache and fast decoding is led by the teaching "
                    "assistant Sara Karimi. The week 10 session on retrieval-augmented generation is led by "
                    "Dr. Navid Rahimi.",
    "office-hours": "Sara Karimi holds office hours on Tuesdays from 14:00 to 16:00 in room 305. Ali Moradi holds "
                    "office hours on Sundays from 10:00 to 12:00 in room 112.",
    "llm-project": "The LLM-501 final project is done in teams of two and must be built with LangGraph or Deep "
                   "Agents. Proposals are due in week 8, and the final report is due one week after the last session.",
    "llm-late-policy": "LLM-501 late policy: a late submission loses 10% of its grade per day, for up to three days. "
                       "After three days it receives zero.",
    "nlp-syllabus": "NLP-402 (Natural Language Processing) is taught by Dr. Leila Hosseini. Grading: homework 30%, "
                    "midterm 30%, final exam 40%.",
    "nlp-late-policy": "NLP-402 late policy: late homework loses 20% of its grade per day and is not accepted after "
                       "two days.",
    "nlp-schedule": "In NLP-402, the sessions on word embeddings (week 4) and on attention (week 7) are led by the "
                    "teaching assistant Ali Moradi.",
    "prerequisites": "LLM-501 requires NLP-402 or an equivalent course. Students without it need written permission "
                     "from the instructor of LLM-501.",
    "exams": "The LLM-501 final exam is on 2026-12-20 in hall A. The NLP-402 final exam is on 2026-12-27 in hall B. "
             "A student who misses a final exam for medical reasons may take a make-up exam within two weeks, with "
             "a medical certificate.",
    "compute": "Each LLM-501 project team gets 20 GPU hours on the department cluster. Requests for more hours go to "
               "the cluster manager, Ali Moradi.",
    "nlp-lab": "Room 112 is the NLP lab. It is directed by Dr. Leila Hosseini, and Ali Moradi is its lab engineer.",
    # new in this tutorial
    "cv-syllabus": "CV-310 (Computer Vision) is taught by Dr. Parisa Ahmadi, with Reza Tavakoli as teaching "
                   "assistant. Grading: two projects 50%, final exam 50%. CV-310 covers convolutional networks "
                   "and vision transformers.",
    "cv-compute": "CV-310 project teams also use the department cluster, with 10 GPU hours per team.",
    "research-groups": "Sara Karimi is a PhD student in Dr. Navid Rahimi's research group, which works on retrieval "
                       "and language agents. Reza Tavakoli is a PhD student supervised by Dr. Parisa Ahmadi.",
    "vision-lab": "Room 210 is the vision lab, directed by Dr. Parisa Ahmadi. Reza Tavakoli holds office hours there "
                  "on Mondays from 9:00 to 11:00.",
}

docs = [Document(page_content=text, metadata={"source": src}) for src, text in HANDBOOK.items()]

# Tutorial 6's baseline, for comparison
text_store = InMemoryVectorStore.from_documents(docs, embeddings)
retriever = text_store.as_retriever(search_kwargs={"k": 2})


def format_docs(docs):
    return "\n\n".join(f"[{d.metadata['source']}] {d.page_content}" for d in docs)


rag_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "Answer the question using only the context below. If the context does not contain the answer, "
     "say so. Do not guess.\n\nContext:\n{context}"),
    ("human", "{question}"),
])
answer_chain = rag_prompt | model | StrOutputParser()


def vector_rag(question):
    found = retriever.invoke(question)
    return answer_chain.invoke({"context": format_docs(found), "question": question}), found


print(len(docs), "documents")

## 3. A knowledge graph by hand

Before asking an LLM to build a graph, build a tiny one yourself, so the data structure is clear.

In [ ]:
import networkx as nx

toy = nx.MultiDiGraph()
toy.add_edge("Navid Rahimi", "LLM-501", relation="TEACHES")
toy.add_edge("LLM-501", "KV cache", relation="COVERS")
toy.add_edge("Sara Karimi", "KV cache", relation="LEADS_SESSION_ON", detail="LLM-501, week 9")
toy.add_edge("Sara Karimi", "room 305", relation="HOLDS_OFFICE_HOURS_IN", detail="Tuesdays 14:00 to 16:00")


def fmt_edge(u, v, data):
    """One edge as a readable triple: (u) -[RELATION {detail}]-> (v)."""
    detail = f" {{{data['detail']}}}" if data.get("detail") else ""
    return f"({u}) -[{data['relation']}{detail}]-> ({v})"


print("All facts:")
for u, v, data in toy.edges(data=True):
    print("  ", fmt_edge(u, v, data))

print("\nFacts about Sara Karimi (outgoing):")
for u, v, data in toy.out_edges("Sara Karimi", data=True):
    print("  ", fmt_edge(u, v, data))

print("\nHow is Navid Rahimi connected to room 305?")
print("  ", " → ".join(nx.shortest_path(toy.to_undirected(), "Navid Rahimi", "room 305")))

### Code walkthrough: nodes, edges, attributes

**A node is an entity, identified by its name.** `add_edge` creates both endpoints if they do not exist yet. That is
convenient and dangerous: "Sara Karimi" and "sara karimi" would silently become two different people. Section 5 is
about exactly that.

**An edge is one fact, and its attributes carry the details.** `relation` says what kind of fact it is; `detail`
holds the part that is not an entity ("Tuesdays 14:00 to 16:00"). Without `detail`, the graph would know *where*
Sara's office hours are but not *when*.

**`MultiDiGraph`** allows several edges between the same two nodes (a person can both teach and coordinate a
course) and keeps their direction (Sara leads a session on the KV cache, not the other way round).

**`to_undirected()` for paths.** Direction matters for meaning but not for connection. Rahimi teaches LLM-501, which
covers the KV cache, which Sara leads, who sits in room 305: following the edges forwards only, there is no path from
Rahimi to room 305 at all. The path is the multi-hop answer, found without a single embedding.

## 4. Extracting the graph with an LLM

Writing triples by hand does not scale. The standard approach is to give an LLM each chunk and a **schema**: the
entity types and relation types it may use. A schema makes the output consistent enough to join up across documents.
Without one, the same fact comes back as `TEACHES`, `IS_INSTRUCTOR_OF` and `LECTURES`, and nothing connects.

In [ ]:
from typing import Literal
from pydantic import BaseModel, Field

EntityType = Literal["Person", "Course", "Topic", "Room", "Resource"]
RelationType = Literal["TEACHES", "ASSISTS", "LEADS_SESSION_ON", "COVERS", "REQUIRES", "HOLDS_OFFICE_HOURS_IN",
                       "DIRECTS", "WORKS_IN", "MANAGES", "USES", "HAS_EXAM_IN", "SUPERVISES"]


class Entity(BaseModel):
    name: str = Field(description="Canonical name. People: full name without titles ('Navid Rahimi'). "
                                  "Courses: the course code ('LLM-501'). Rooms: 'room 305', 'hall A'.")
    type: EntityType


class Relation(BaseModel):
    source: str = Field(description="Name of the source entity, exactly as written in entities.")
    relation: RelationType
    target: str = Field(description="Name of the target entity, exactly as written in entities.")
    detail: str = Field(default="", description="Extra facts that are not entities: a day and time, a date, "
                                                "a week, a number of hours. Empty if there are none.")


class Extraction(BaseModel):
    entities: list[Entity]
    relations: list[Relation]


EXTRACT_PROMPT = """Extract a knowledge graph from the text below, using only these relations, in this direction:

- TEACHES: Person -> Course (the instructor)
- ASSISTS: Person -> Course (a teaching assistant)
- LEADS_SESSION_ON: Person -> Topic (detail: course and week)
- COVERS: Course -> Topic
- REQUIRES: Course -> Course (a prerequisite)
- HOLDS_OFFICE_HOURS_IN: Person -> Room (detail: day and time)
- DIRECTS: Person -> Room or Resource
- WORKS_IN: Person -> Room
- MANAGES: Person -> Resource
- USES: Course -> Resource (detail: allowance, such as GPU hours)
- HAS_EXAM_IN: Course -> Room (detail: date)
- SUPERVISES: Person -> Person (research supervisor of a student)

Leave out facts that fit none of these relations. Every source and target must appear in entities.

Text:
{text}"""

extractor = model.with_structured_output(Extraction)

example = extractor.invoke(EXTRACT_PROMPT.format(text=HANDBOOK["llm-schedule"]))
for e in example.entities:
    print(f"  {e.type:<9} {e.name}")
for r in example.relations:
    print("  ", fmt_edge(r.source, r.target, r.model_dump()))

### Code walkthrough: the schema is the design

**`Literal[...]` for types and relations.** The model cannot invent a relation outside the list, so the graph has a
fixed vocabulary that your queries and tools can rely on. Choosing that vocabulary is the most important design
decision in GraphRAG, and it is yours, not the model's: write down the questions you need to answer, then the
relations those questions walk along.

**The prompt gives each relation a direction.** "TEACHES: Person -> Course" stops half the edges from coming out
backwards. A graph with inconsistent directions still has correct paths, but queries such as "who teaches X?" (the
*incoming* `TEACHES` edges of X) return nothing.

**The `name` description is a first line of defence against duplicates.** Asking for "full name without titles"
and "the course code" means "Dr. Rahimi", "Navid Rahimi" and "the instructor of LLM-501" have a chance of
becoming one node. Only a chance: section 5 cleans up the rest.

**What is deliberately not extracted.** Grading percentages and late policies fit no relation, so they are left out.
That is not a bug. A rule like "loses 10% per day for three days" is prose, and vector search over the original text
handles it better than any triple. Section 10 uses both.

### Extract the whole handbook

One call per document. With 16 short documents this takes under a minute; for a real corpus it is the dominant cost
of GraphRAG, and the reason to keep the results (`extractions`) instead of re-running the step.

In [ ]:
import time

extractions = {}
for src, text in HANDBOOK.items():
    extractions[src] = extractor.invoke(EXTRACT_PROMPT.format(text=text))
    print(f"{src:<16} {len(extractions[src].entities):>2} entities, {len(extractions[src].relations):>2} relations")
    time.sleep(1)                      # be gentle with free-tier rate limits


def build_graph(extractions, resolve=lambda name: name):
    """Turn the extractions into one graph. `resolve` maps every name to its canonical form (section 5)."""
    G = nx.MultiDiGraph()
    for src, ex in extractions.items():
        for e in ex.entities:
            G.add_node(resolve(e.name), type=e.type)
        for r in ex.relations:
            G.add_edge(resolve(r.source), resolve(r.target),
                       relation=r.relation, detail=r.detail, source=src)
    return G


G_raw = build_graph(extractions)
print(f"\nraw graph: {G_raw.number_of_nodes()} nodes, {G_raw.number_of_edges()} edges")

### Code walkthrough: provenance

**`source=src` on every edge** records which document the fact came from. It is the graph's version of the
`[source]` tags in tutorial 6: when an answer is built from graph facts, you can still point to the text, and when a
fact is wrong, you know which document (and which extraction call) to look at.

**`extractions` is kept separate from the graph.** The graph is rebuilt from it in section 5, with names cleaned up,
without calling the model again. In a real project you would save `extractions` to disk (it is a dict of Pydantic
objects, so `model_dump()` gives you JSON) and re-extract only documents that changed.

## 5. Entity resolution: one thing, one node

Look at the node names. Each document was extracted independently, so the same entity can come back spelled
differently in different documents, and each spelling becomes its own node, with its own facts. A graph like that
cannot answer "how is A connected to B?" because A is three disconnected pieces.

In [ ]:
import difflib

print("nodes:")
for name, data in sorted(G_raw.nodes(data=True)):
    print(f"  {data.get('type', '(no type)'):<10} {name}")

print("\nnames that look alike:")
names = sorted(G_raw.nodes)
for a in names:
    for b in difflib.get_close_matches(a, names, n=4, cutoff=0.7):
        if a < b:
            print(f"  {a!r:<32} ~ {b!r}")

### Code walkthrough: what to look for

Your list depends on what the model extracted, but the usual suspects are:

- **titles and case**: "Dr. Leila Hosseini" next to "Leila Hosseini", or "Room 112" next to "room 112";
- **long and short forms**: "LLM-501" next to "Large Language Models" or "LLM-501 (Large Language Models)";
- **descriptions instead of names**: "department cluster", "the cluster", "GPU cluster";
- **`(no type)` nodes**: a name used in a relation that was never listed as an entity, usually a spelling variant of
  one that was.

`difflib.get_close_matches` only finds names that are spelled alike. It will flag "room 112" ~ "room 210" (wrong:
different rooms) and miss "LLM-501" ~ "Large Language Models" (right: same course). That is why the next cell uses
rules for the predictable variants and a hand-written alias table for the rest. Production systems add embedding
similarity and an LLM judge, but still keep a human-reviewed alias list.

In [ ]:
import re

TITLES = re.compile(r"^(dr|prof|professor|mr|mrs|ms)\.?\s+", re.IGNORECASE)

# lower-case variant -> canonical name. Extend this after reading the list above.
ALIASES = {
    "large language models": "LLM-501",
    "llm-501 (large language models)": "LLM-501",
    "llm course": "LLM-501",
    "natural language processing": "NLP-402",
    "nlp-402 (natural language processing)": "NLP-402",
    "computer vision": "CV-310",
    "cv-310 (computer vision)": "CV-310",
    "the department cluster": "department cluster",
    "cluster": "department cluster",
    "gpu cluster": "department cluster",
    "nlp lab": "room 112",
    "the nlp lab": "room 112",
    "vision lab": "room 210",
    "the vision lab": "room 210",
}

canonical_form = {}       # lower-case key -> the display name used for the node


def resolve(name):
    clean = re.sub(r"\s+", " ", TITLES.sub("", name.strip()))
    clean = ALIASES.get(clean.lower(), clean)
    return canonical_form.setdefault(clean.lower(), clean)


G = build_graph(extractions, resolve)

print(f"raw graph:      {G_raw.number_of_nodes()} nodes")
print(f"resolved graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")
print("untyped nodes left:", [n for n, d in G.nodes(data=True) if "type" not in d])

### Code walkthrough: rules first, then a table

**`resolve` is applied to every name, in entities and in relations,** before anything is added to the graph. Doing it
at build time, rather than merging nodes afterwards, means edges never point at a variant that no longer exists.

**Three steps, cheapest first.** Strip titles and extra spaces (a rule), look the lower-cased name up in `ALIASES`
(a table), then use the lower-cased result as the key, so case differences never create a new node (a rule). Rules handle the predictable variation; the table handles knowledge that no rule has,
such as "the NLP lab" being "room 112".

**`canonical_form` keeps the first spelling seen** for each key, so nodes keep readable names ("Navid Rahimi", not
"navid rahimi"), while every variant maps to the same node.

**Entity resolution is where GraphRAG quality is decided.** A missed merge splits a person in two and breaks every
path through them; a wrong merge (two different rooms becoming one) creates connections that do not exist, which is
worse, because the answers built on them look confident. Re-run the previous cell on `G` instead of `G_raw` and keep
extending `ALIASES` until the remaining look-alikes are genuinely different things.

## 6. Look at the graph

A drawing is the fastest way to spot extraction problems: an isolated node, a person with no course, a relation
pointing the wrong way.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

COLORS = {"Person": "#4C72B0", "Course": "#DD8452", "Topic": "#55A868", "Room": "#C44E52", "Resource": "#8172B3"}


def draw(G, title=""):
    H = nx.DiGraph(G)                       # one arrow per pair of nodes, for a readable picture
    pos = nx.spring_layout(H, seed=7, k=1.2)
    plt.figure(figsize=(14, 10))
    nx.draw_networkx_nodes(H, pos, node_size=1100, alpha=0.9,
                           node_color=[COLORS.get(H.nodes[n].get("type"), "#BBBBBB") for n in H])
    nx.draw_networkx_labels(H, pos, font_size=8)
    nx.draw_networkx_edges(H, pos, alpha=0.35, arrowsize=12, node_size=1100)
    nx.draw_networkx_edge_labels(H, pos, font_size=6,
                                 edge_labels={(u, v): d["relation"] for u, v, d in H.edges(data=True)})
    plt.legend(handles=[Patch(color=c, label=t) for t, c in COLORS.items()], loc="lower left")
    plt.title(title)
    plt.axis("off")
    plt.show()


draw(G, "The department handbook as a knowledge graph")

print("Most connected entities:")
U = nx.Graph(G.to_undirected())
for name, degree in sorted(U.degree, key=lambda x: -x[1])[:6]:
    print(f"  {degree:>2}  {name} ({G.nodes[name].get('type', '?')})")

### Code walkthrough: degree is already an answer

**`nx.DiGraph(G)`** collapses parallel edges into one arrow per pair, only for drawing. The real graph `G` keeps them.

**Degree** is the number of distinct neighbours. "Who is involved in the most things?" is a degree question, and it
is one that vector RAG cannot answer at all: no chunk lists all of Ali Moradi's roles, and retrieving two chunks
cannot count the rest. The graph answers it with no model call.

Things to check in the picture: are the three courses each connected to their instructor and teaching assistant?
Does anything float on its own? Is `Ali Moradi` (who appears in four documents) one node or several?

## 7. Local search: start from the entities in the question

**Local search** answers questions about specific entities. Three steps:

```text
question → (1) link: which nodes does it mention? → (2) expand: their k-hop neighbourhood
         → (3) answer from those facts, written as text
```

Step 1 is **entity linking**. The question says "the cluster manager" or "the KV cache lecture", not the node names,
so an LLM pulls out the mentions and embeddings match them to nodes.

In [ ]:
class Mentions(BaseModel):
    names: list[str] = Field(description="People, courses, topics, rooms or resources mentioned in the question, "
                                         "as short noun phrases.")


mention_extractor = model.with_structured_output(Mentions)
node_index = InMemoryVectorStore.from_texts(list(G.nodes), embeddings)


def link_entities(question, threshold=0.55):
    mentions = mention_extractor.invoke(f"List the entities mentioned in this question: {question}").names
    linked = []
    for mention in mentions:
        node, score = node_index.similarity_search_with_score(mention, k=1)[0]
        print(f"   {mention!r:<28} → {node.page_content!r} ({score:.2f})")
        if score >= threshold:
            linked.append(node.page_content)
    return linked


def subgraph_facts(G, seeds, hops=2):
    U = G.to_undirected(as_view=True)
    keep = set()
    for seed in seeds:
        keep |= set(nx.ego_graph(U, seed, radius=hops))
    return [fmt_edge(u, v, d) for u, v, d in G.subgraph(keep).edges(data=True)]


GRAPH_PROMPT = """Answer the question using only the facts below, written as (subject) -[RELATION {{detail}}]-> (object).
Follow chains of facts when needed. If the facts do not contain the answer, say so. Do not guess.

Facts:
{facts}

Question: {question}"""


def graph_rag(question, hops=2):
    seeds = link_entities(question)
    facts = subgraph_facts(G, seeds, hops)
    answer = model.invoke(GRAPH_PROMPT.format(facts="\n".join(facts), question=question)).content
    return answer, seeds, facts


question = "Who directs the lab where the cluster manager holds office hours?"
print("vector RAG:", vector_rag(question)[0].replace("\n", " "), "\n")

for hops in [1, 2, 3]:
    print(f"graph RAG, {hops} hop(s):")
    answer, seeds, facts = graph_rag(question, hops)
    print(f"   {len(facts)} facts → {answer.strip()[:200]}\n")

### Code walkthrough: linking, radius and the facts prompt

**The question is three hops long**: cluster ← managed by Ali Moradi → office hours in room 112 ← directed by Leila
Hosseini. Vector RAG with `k=2` sees at most two of the three documents involved, and nothing in the question
mentions Leila Hosseini or room 112.

**Read the linking lines.** "the cluster manager" should link to `department cluster`. "the lab" usually scores
below the threshold and is dropped: no node is *called* a lab, because entity resolution named it `room 112`. Linking
matches names, so knowledge that lives only in `ALIASES` is invisible to it (exercise 10.1 fixes that). The
`threshold` decides what counts as a match: too high and the search has no starting point, too low and unrelated
nodes join in.

**`hops` is the radius around each linked entity.** One hop sees Ali Moradi but not the room; two hops reach room
112; three reach its director. Every extra hop multiplies the number of facts, and in a real graph a well-connected
node (a department, a popular topic) can pull in thousands. Production systems rank and cut the facts before the
prompt, or let an agent choose which edges to follow, which is the next section.

**Facts are serialised as text.** The model never sees the graph object, only lines like
`(Ali Moradi) -[HOLDS_OFFICE_HOURS_IN {Sundays 10:00 to 12:00}]-> (room 112)`. The graph's job is to choose *which*
facts reach the prompt. The doubled braces in `GRAPH_PROMPT` are there because `.format()` would otherwise read
`{detail}` as a placeholder.

## 8. An agent with graph tools

A fixed radius is a blunt instrument. Tutorial 6 handed retrieval decisions to an agent; here the same idea applies
to the graph: give an agent a few small tools and let it walk the graph one step at a time, choosing which edges to
follow.

In [ ]:
from langchain.agents import create_agent
from langchain_core.tools import tool


@tool
def find_entities(name: str) -> str:
    """Find entities in the department knowledge graph whose name is close to `name`.
    Call this first: the other tools need exact entity names."""
    hits = node_index.similarity_search_with_score(name, k=3)
    return "\n".join(f"{d.page_content} ({G.nodes[d.page_content].get('type', '?')}), similarity {s:.2f}"
                     for d, s in hits)


@tool
def get_relations(entity: str) -> str:
    """List every fact that involves one entity, in both directions. `entity` must be an exact name."""
    if entity not in G:
        return f"No entity named {entity!r}. Use find_entities first."
    edges = list(G.out_edges(entity, data=True)) + list(G.in_edges(entity, data=True))
    return "\n".join(fmt_edge(u, v, d) for u, v, d in edges) or "No facts."


@tool
def find_path(start: str, end: str) -> str:
    """Show how two entities are connected: the chain of facts on the shortest path between them.
    Both names must be exact."""
    for name in (start, end):
        if name not in G:
            return f"No entity named {name!r}. Use find_entities first."
    try:
        path = nx.shortest_path(G.to_undirected(as_view=True), start, end)
    except nx.NetworkXNoPath:
        return f"{start} and {end} are not connected."
    facts = []
    for a, b in zip(path, path[1:]):
        u, v = (a, b) if G.has_edge(a, b) else (b, a)
        facts.append(fmt_edge(u, v, next(iter(G.get_edge_data(u, v).values()))))
    return "\n".join(facts)


@tool
def rank_entities(entity_type: str) -> str:
    """Rank all entities of one type (Person, Course, Topic, Room or Resource) by how many facts they
    take part in. Use it for questions such as 'who is involved in the most...'."""
    ranked = sorted((n for n, d in G.nodes(data=True) if d.get("type") == entity_type),
                    key=lambda n: -G.degree(n))
    return "\n".join(f"{n}: {G.degree(n)} facts" for n in ranked) or f"No entities of type {entity_type!r}."


GRAPH_TOOLS = [find_entities, get_relations, find_path, rank_entities]

graph_agent = create_agent(
    model,
    tools=GRAPH_TOOLS,
    system_prompt=(
        "You answer questions about a university department using a knowledge graph. "
        "Use find_entities to get exact names, then get_relations, find_path or rank_entities. "
        "Answer only from the facts the tools return, and say which facts you used. "
        "If the graph does not contain the answer, say so."
    ),
    name="graph_agent",
)

for question in ["Which person in the department has the most roles, and what are they?",
                 "How is Navid Rahimi connected to room 305?"]:
    print(f"===== {question}")
    show_trace(ask(graph_agent, question)["messages"])
    print()

### Code walkthrough: tools are graph queries

Each tool is one small, safe query: neighbours, shortest path, ranking by degree. Together they cover most of what
students ask, and the agent combines them: `rank_entities("Person")`, then `get_relations` on the winner.

**Exact names are enforced.** `get_relations` and `find_path` refuse unknown names and point the model at
`find_entities`. A tool that silently returned nothing would teach the model that the entity has no facts.

**Why not let the model write graph queries directly?** With a graph database, a common design is *text-to-Cypher*:
the model writes a query in the database's language and you run it (section 11). That is more flexible and much
riskier: a query the model gets subtly wrong returns a confident wrong answer, and a malicious question can try to
write to or delete from the database. A handful of fixed tools is the safer default; give the model a query language
only when the tools truly cannot express the questions, and then with a read-only connection.

Compare `rank_entities` with tutorial 6: no amount of text retrieval answers "who has the most roles", because the
answer is a count over the whole corpus. That is the kind of question that justifies building a graph.

### Exercise 8.1

1. Ask the graph agent "What does the LLM-501 late policy say?". The fact is in the handbook but not in the graph.
   Does the agent admit that, or does it make something up from nearby facts?
2. Add a tool `facts_by_relation(relation: str)` that returns every edge of one relation type (all `TEACHES` edges,
   say). Which questions become easy that were hard before?

## 9. Global search: questions about everything

Some questions are about the corpus as a whole: "What are the main groups in this department and what does each
do?" There is no entity to start from, so local search has nothing to link to, and vector search returns two
chunks, which is a sample, not a summary.

Microsoft's GraphRAG answers these with **communities**:

```text
graph → (1) split into communities: densely connected groups of nodes
      → (2) write a short report on each community        (once, at indexing time)
      → (3) at question time: ask each report for its relevant points (map),
            then combine the points into one answer (reduce)
```

In [ ]:
U = nx.Graph(G.to_undirected())
communities = [c for c in nx.community.louvain_communities(U, seed=42) if len(c) > 1]

for i, members in enumerate(communities):
    print(f"community {i}: {sorted(members)}")

### Code walkthrough: what a community is

**`louvain_communities`** groups nodes so that there are many edges inside each group and few between groups. Nobody
told it about courses or research groups; it finds them from the shape of the graph alone. On this graph you will
usually see something close to "the LLM course and its people", "the NLP course and its lab", "the vision course
and its lab", with shared nodes such as the cluster placed in whichever group they are most tied to.

**`seed=42`** makes the result reproducible. Louvain is randomised, and different seeds can move border nodes
between communities.

Real GraphRAG systems use the **Leiden** algorithm and build a *hierarchy* of communities (big groups split into
smaller ones), so a question can be answered at the level of detail it needs. One level is enough to see the idea.

In [ ]:
class Report(BaseModel):
    title: str = Field(description="A short name for the group, e.g. 'NLP teaching team'.")
    summary: str = Field(description="Two or three sentences: who and what is in the group, and how they connect.")


reporter = model.with_structured_output(Report)

reports = []
for members in communities:
    facts = [fmt_edge(u, v, d) for u, v, d in G.subgraph(members).edges(data=True)]
    report = reporter.invoke("Write a short report on this group from a university department's knowledge "
                             "graph.\n\nFacts:\n" + "\n".join(facts))
    reports.append(report)
    print(f"## {report.title}\n{report.summary}\n")

In [ ]:
class Partial(BaseModel):
    useful: bool = Field(description="True if the report contains anything that helps answer the question.")
    points: str = Field(description="The points from the report that help answer the question. Empty if none.")


mapper = model.with_structured_output(Partial)


def global_search(question):
    notes = []
    for report in reports:                                             # map
        partial = mapper.invoke(f"Question: {question}\n\nReport: {report.title}\n{report.summary}")
        if partial.useful:
            notes.append(f"- {report.title}: {partial.points}")
    return model.invoke(                                               # reduce
        "Combine these notes into one well-organised answer to the question. Use only the notes.\n\n"
        f"Question: {question}\n\nNotes:\n" + "\n".join(notes)
    ).content


question = "What are the main groups of people in this department, and what does each group work on?"
print("vector RAG:\n", vector_rag(question)[0], "\n")
print("global search:\n", global_search(question))

### Code walkthrough: map-reduce over reports

**Reports are written once, at indexing time.** At question time the model reads a few paragraphs instead of the
whole corpus. That is what makes "summarise everything" affordable when "everything" is thousands of documents.

**Map: one call per report, in isolation.** Each call answers "what in *this* report helps?" and may say nothing
helps. With a handful of reports you could put them all in one prompt; map-reduce is the version that still works
when the reports do not fit in a context window, and the map calls can run in parallel.

**Reduce: one call to combine.** It sees only the useful points, already filtered.

**Compare the two answers.** Vector RAG describes whatever its two chunks happened to cover, usually one course.
Global search should name all three groups. That difference is the whole argument for community summaries, and the
cost is visible too: the reports took one call per community to write, and the question took one call per report
plus one.

## 10. Hybrid: let an agent choose text or graph

Neither store is enough on its own. The graph has no grading percentages or late policies, because they were never
triples; the text store cannot count, follow long chains, or summarise everything. Give one agent both, with
docstrings that say what each is for, and let it route, which is tutorial 6's idea with more than one kind of
retriever.

In [ ]:
@tool
def search_handbook(query: str) -> str:
    """Search the full text of the handbook. Best for rules, policies, grading, deadlines and exact
    wording. Use a short query about one thing."""
    return format_docs(retriever.invoke(query))


@tool
def department_overview() -> str:
    """Return short reports on the main groups in the department. Use it for broad questions about
    the department as a whole."""
    return "\n\n".join(f"{r.title}: {r.summary}" for r in reports)


hybrid_agent = create_agent(
    model,
    tools=[search_handbook, department_overview, *GRAPH_TOOLS],
    system_prompt=(
        "You answer questions about a university department. You have three kinds of tools:\n"
        "- search_handbook: the handbook text, for rules, policies, grading and deadlines;\n"
        "- find_entities, get_relations, find_path, rank_entities: a knowledge graph of people, courses, topics, "
        "rooms and resources, for connections, multi-step links and counting;\n"
        "- department_overview: summaries of the main groups, for questions about the department as a whole.\n"
        "Pick the tool that fits each part of the question. Answer only from what the tools return."
    ),
    name="hybrid_agent",
)

for question in [
    "What percentage of the CV-310 grade is the final exam?",                         # text
    "I missed the KV cache lecture. Where and when can I find the person who gave it?",  # graph
    "Give me a short overview of who does what in this department.",                 # global
]:
    print(f"===== {question}")
    show_trace(ask(hybrid_agent, question)["messages"])
    print()

### Code walkthrough: routing by docstring

**The three questions are chosen to need three different tools.** Read the traces and check that the agent picked
`search_handbook` for the grading question, the graph tools for the KV cache question, and `department_overview` for
the last. When it picks wrong, the fix is almost always a clearer docstring or system prompt, not more code.

**The KV cache question now takes two graph calls** (`find_entities`, `get_relations`), or one `find_path`, instead
of tutorial 6's two searches and two full RAG runs, and the connection it follows was checked once, at indexing
time, instead of being rediscovered from text on every question.

**Six tools is near the comfortable limit** for a small model. Past that, tool choice gets noticeably worse, and
tutorial 4's answer applies: split into specialist agents (a text agent and a graph agent) under a supervisor.

### Exercise 10.1

1. Ask the hybrid agent "Which courses use the department cluster, and how many GPU hours does each team get?".
   Does it use the graph, the text, or both? Which source has the hour numbers?
2. Make entity linking see aliases: build `node_index` from `list(G.nodes) + list(ALIASES)`, with
   `metadatas=[{"node": resolve(n)} for n in ...]`, and return `node.metadata["node"]` in `link_entities`. Does "the
   lab" link now, and to which lab? How many hops does section 7's question need after the change?
3. Remove `ALIASES` (set it to `{}`), rebuild `G`, `node_index` and the reports, and re-run section 7's question.
   Which step breaks first?
4. Add a document: "From week 11, Sara Karimi also assists in CV-310." Extract it, add its relations to `G` with
   `resolve`, and ask "Which teaching assistants work on more than one course?". What would it take to answer this with
   vector RAG alone?

## 11. Choosing, and where to go next

| Question type | Best retriever | Why |
|---|---|---|
| A rule, a policy, an explanation | **Vector** (tutorial 6) | The answer is a passage of text |
| A fact about one entity | Either | A single chunk or a single edge both work |
| A chain of facts across documents | **Graph**, local search or tools | Paths are stored, not rediscovered |
| "Who/which ... the most", "how many" | **Graph** tools | Counting needs the whole corpus |
| "What is this corpus about?" | **Graph** global search | Community reports summarise everything |
| A mix | **Hybrid** agent | Route each part to the store that fits |

**When a graph is worth it.** Your corpus is about entities and their relations (people, products, contracts,
proteins, regulations), questions often cross documents, and the corpus changes slowly enough to re-extract. If your
users mostly ask "what does the document say about X?", a good vector RAG pipeline with tutorial 6's grading is
cheaper and simpler.

**What it costs.** One or more LLM calls per chunk at indexing time, plus community reports; schema design; entity
resolution, which never quite ends; and re-extraction when documents change. Extraction errors are silent: a missed
relation is simply an answer the system cannot give.

### Going further

- **Graph databases.** NetworkX lives in memory. For a real corpus, store the graph in a database such as **Neo4j**
  and query it with **Cypher**. LangChain's `langchain-neo4j` package provides `Neo4jGraph` to load triples and run
  queries, and `GraphCypherQAChain` for text-to-Cypher (always with a read-only user):

  ```python
  # pip install langchain-neo4j   (needs a running Neo4j instance)
  from langchain_neo4j import Neo4jGraph
  graph = Neo4jGraph(url="bolt://localhost:7687", username="neo4j", password="...")
  graph.query("MATCH (p:Person)-[:HOLDS_OFFICE_HOURS_IN]->(r:Room) RETURN p.name, r.name")
  ```

- **Ready-made extraction.** `LLMGraphTransformer` (in `langchain-experimental`) does section 4's job with
  configurable node and relation types, and writes straight into Neo4j.
- **Microsoft GraphRAG** (`pip install graphrag`) is the full version of sections 4 to 9: extraction, Leiden
  community hierarchies, community reports, and local and global search, driven by configuration files.
- **LightRAG** and similar projects combine vector and graph retrieval in one index, the idea of section 10 built in.

## Glossary: quick reference

| Term | Meaning |
|---|---|
| **Knowledge graph** | A graph whose nodes are entities and whose edges are typed relations between them. |
| **Triple** | One fact: (subject, relation, object), such as (Sara Karimi, HOLDS_OFFICE_HOURS_IN, room 305). |
| **Schema** | The allowed entity types and relation types, with directions. Keeps extraction consistent. |
| **Graph extraction** | Using an LLM to turn text into entities and relations that fit the schema. |
| **Provenance** | A link from each fact back to the document it came from. |
| **Entity resolution** | Merging the different names of one real thing into a single node. |
| **Entity linking** | Matching the mentions in a question to nodes in the graph. |
| **k-hop neighbourhood** | All nodes within k edges of a starting node (`nx.ego_graph`). |
| **Local search** | Answering from the facts around the entities a question mentions. |
| **Degree** | The number of neighbours of a node; a simple measure of how connected it is. |
| **Community** | A densely connected group of nodes, found by an algorithm such as Louvain or Leiden. |
| **Community report** | An LLM-written summary of one community, made at indexing time. |
| **Global search** | Answering whole-corpus questions by map-reduce over community reports. |
| **Text-to-Cypher** | Letting the model write graph database queries; flexible, and risky without a read-only connection. |
| **Hybrid RAG** | Combining vector and graph retrieval, usually with an agent choosing between them. |

---